# Qwen2.5-3B Fine-tuning — CPT then SFT for Knowledge Injection

In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

UsageError: Line magic function `%%capture` not found.


In [2]:
%pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 26.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 33.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 107.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 119.0 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 10.2 MB/s eta 0:0

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [10]:
# ── Cell 3: Paths & Config ───────────────────────────────────────────────────
BASE_DIR    = "/kaggle/input/datasets/sasidhar77/hacaton-dataset"
CKT_DIR = "/kaggle/working/"
DATA_PATH   = f"{BASE_DIR}/final_training_file.json"
RAW_TEXT    = f"{BASE_DIR}/cpt_loan_domain.txt"

CPT_DIR     = f"{CKT_DIR}/qwen3_v4_cpt_ckpts"
SFT_DIR     = f"{CKT_DIR}/qwen3_v4_sft_ckpts"
MERGED_DIR  = f"{CKT_DIR}/qwen3_v4_merged"

print(f"Raw text: {RAW_TEXT}")
print(f"Q&A:      {DATA_PATH}")
print(f"CPT out:  {CPT_DIR}")
print(f"SFT out:  {SFT_DIR}")
print(f"Merged:   {MERGED_DIR}")

Raw text: /kaggle/input/datasets/sasidhar77/hacaton-dataset/cpt_loan_domain.txt
Q&A:      /kaggle/input/datasets/sasidhar77/hacaton-dataset/final_training_file.json
CPT out:  /kaggle/working//qwen3_v4_cpt_ckpts
SFT out:  /kaggle/working//qwen3_v4_sft_ckpts
Merged:   /kaggle/working//qwen3_v4_merged


In [5]:
# ── Cell 4: Load Model ───────────────────────────────────────────────────────
# Using Instruct variant (Qwen3-4B-Instruct-2507). For CPT-then-SFT, the Base
# variant is sometimes cleaner — but Instruct works fine if the LoRA has
# enough capacity (r=64 below) to override the 'be generic' prior.
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Qwen2.5-3B-Instruct",
    max_seq_length  = 2048,
    load_in_4bit    = True,
    load_in_8bit    = False,
    full_finetuning = False,
)

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM: {gpu.total_memory/1024**3:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
GPU: Tesla T4 | VRAM: 14.6 GB


In [6]:
# ── Cell 5: LoRA Adapter (used for BOTH CPT and SFT — same adapter) ──────────
#
# Why r=64 alpha=128:
#   - r=32 in v3 wasn't enough capacity to (a) memorize a 46K-token report AND
#     (b) override the Instruct model's strong 'give a generic answer' prior.
#   - r=64 doubles capacity. With rsLoRA, effective scale = alpha/sqrt(r) = 128/8 = 16,
#     which is meaningful but stable.
#   - Targeting all 7 linear modules (attn + MLP) is correct; embed_tokens/lm_head
#     are NOT included on purpose — adding them on a 4B Instruct model with so
#     little data destabilizes generation.
#
# Why NOT use_dora here:
#   DoRA + rsLoRA + Unsloth has known compatibility issues. Standard LoRA with
#   rsLoRA is well-tested and gives essentially the same generalization for
#   knowledge-injection tasks.
#
# Why dropout=0.05:
#   For knowledge memorization (CPT), we WANT memorization; high dropout
#   actively hurts. Keep it low.

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
    use_rslora     = True,
    loftq_config   = None,
)
model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [7]:
# ── Cell 6: Chat Template ────────────────────────────────────────────────────
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

In [8]:
# ── Cell 7: Inference helper — baseline test on loan decision examples ────────
PROBE_EXAMPLES = [
    {
        "prompt": "[TASK] Recommend a loan approval decision: Approve | Approve with Conditions | Reject\n\n[APPLICANT]\nAge / Occupation : 65 yrs | Self Employed\nMonthly Income   : ₹93,318 | Disposable ₹0\n\n[LOAN REQUEST]\nProduct          : MSME Loan\nAmount Requested : ₹2,071,034\nTenure           : 24 months | Est. EMI ₹101,620\nFOIR             : 1.00 (Financially Stressed)\n\n[CREDIT PROFILE]\nBureau Score     : 793 (Very Good)\nPayment History  : Severely Delinquent\nCurrent DPD      : 75 days (Serious Delinquency)\nMax DPD          : 75 days (Serious Delinquency)\nCredit Util.     : 31% (Low)\n\n[FLAGS]\nDefault          : No | Write-Off: No\n\n[RECOMMENDATION]",
        "completion": "Reject"
    },
    {
        "prompt": "[TASK] Classify the credit risk of the borrower below.\nOutput one of: LOW RISK | MEDIUM RISK | HIGH RISK | CRITICAL RISK\n\n[BORROWER]\nAge / Gender   : 21 yrs | Female\nOccupation     : Salaried\nMonthly Income : ₹37,094 | Disposable ₹0\n\n[CREDITWORTHINESS]\nBureau Score   : 822 (Excellent)\nFOIR           : 1.00 (Financially Stressed)\nCredit Util.   : 31% (Low)\n\n[REPAYMENT HISTORY]\nCurrent DPD    : 0 days (No Delinquency)\nMax DPD        : 60 days (Moderate Delinquency)\nBucket         : Current\nPayment Trend  : Recovered From Past Delays\n\n[FLAGS]\nLoan Product   : Home Loan\nDefault        : No\nWrite-Off      : No\n\n[CLASSIFICATION]",
        "completion": "MEDIUM RISK\nKey signals: strong bureau score (822 (Excellent)); past delinquency (max 60 days, now recovered); stressed FOIR 1.00. Enhanced monitoring — escalate if DPD or FOIR worsens."
    },
    {
        "prompt": "[TASK] Generate a concise loan summary for a credit analyst.\n\n[BORROWER]\nCustomer       : CUST000056 | Age 44 | Male | Telangana\nOccupation     : Business\nMonthly Income : ₹30,011 | Disposable ₹0\n\n[LOAN]\nLoan ID        : LN0000844 | Home Loan\nSanctioned     : ₹2,842,705 | 180 months | 13.15% p.a.\nEMI            : ₹36,248 | FOIR 1.00 (Financially Stressed)\nOutstanding    : ₹1,068,964 | 38% utilization (Low)\nRepayment Stage: Partially Repaid\n\n[CREDIT]\nBureau Score   : 708 (Good)\nPayment Trend  : Recovered From Past Delays\n\n[DELINQUENCY]\nCurrent DPD    : 0 days (No Delinquency) | Bucket: Current\nMax DPD        : 45 days (Moderate Delinquency)\n\n[STATUS]\nLoan Status    : Active | Default: No | Write-Off: No\nRisk Tier      : MEDIUM | Account Health: Healthy | Risk Driver: High EMI Burden\n\n[SUMMARY]",
        "completion": "LN0000844 | Home Loan | Active | MEDIUM RISK\n\nBORROWER: CUST000056, 44yr male, Telangana. cyclical business income — ₹30,011/mo, disposable ₹0.\n\nLOAN: property-backed Home Loan — ₹2,842,705 over 180 months at 13.15% p.a. (EMI ₹36,248). FOIR 1.00 (Financially Stressed). Outstanding ₹1,068,964 — partially repaid, low utilization (38%). Bureau: 708 — Good.\n\nREPAYMENT: Currently regular (DPD 0); historical Max DPD 45 days (Moderate Delinquency) — Recovered From Past Delays.\n\nRISK: MEDIUM RISK — Flags: FOIR 1.00. Utilization 38% (Low). Enhanced monitoring; escalate if DPD worsens."
    },
]

def ask(prompt: str, max_new_tokens: int = 300) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
            repetition_penalty=1.1,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def probe(label: str):
    FastLanguageModel.for_inference(model)
    print(f"\n{'='*78}\n=== PROBE: {label}\n{'='*78}")
    for i, ex in enumerate(PROBE_EXAMPLES):
        print(f"\n--- Example {i+1} ---")
        print(f"PROMPT:\n{ex['prompt']}")
        print(f"\nEXPECTED:\n{ex['completion']}")
        print(f"\nMODEL OUTPUT:\n{ask(ex['prompt'], 300)}")
        print("-" * 78)
    print("=" * 78)

probe("BASELINE (no training yet)")


=== PROBE: BASELINE (no training yet)

--- Example 1 ---
PROMPT:
[TASK] Recommend a loan approval decision: Approve | Approve with Conditions | Reject

[APPLICANT]
Age / Occupation : 65 yrs | Self Employed
Monthly Income   : ₹93,318 | Disposable ₹0

[LOAN REQUEST]
Product          : MSME Loan
Amount Requested : ₹2,071,034
Tenure           : 24 months | Est. EMI ₹101,620
FOIR             : 1.00 (Financially Stressed)

[CREDIT PROFILE]
Bureau Score     : 793 (Very Good)
Payment History  : Severely Delinquent
Current DPD      : 75 days (Serious Delinquency)
Max DPD          : 75 days (Serious Delinquency)
Credit Util.     : 31% (Low)

[FLAGS]
Default          : No | Write-Off: No

[RECOMMENDATION]

EXPECTED:
Reject


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL OUTPUT:
Based on the provided information, I would recommend **Reject** for this loan application.

### Reasons for Rejection:

1. **High Financial Stress (FOIR 1.00)**:
   - The applicant's financial stress level is very high at 1.00, which indicates that they may be struggling to meet their monthly expenses and debt obligations.
   
2. **Severely Delinquent Payment History**:
   - The current delinquency (DPD) of 75 days and max delinquency (also 75 days) indicate a serious problem in meeting payment commitments. This suggests significant financial difficulties.

3. **Low Credit Utilization Rate (31%)**:
   - While the credit utilization rate is low, it doesn't necessarily mitigate the other issues present in the profile.

4. **No Bureau Score Information Provided**:
   - Although the bureau score is given as 793 (which is "Very Good"), there isn't any additional context or details about how this score was calculated or if it aligns with the overall picture presented by the re

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL OUTPUT:
LOW RISK
------------------------------------------------------------------------------

--- Example 3 ---
PROMPT:
[TASK] Generate a concise loan summary for a credit analyst.

[BORROWER]
Customer       : CUST000056 | Age 44 | Male | Telangana
Occupation     : Business
Monthly Income : ₹30,011 | Disposable ₹0

[LOAN]
Loan ID        : LN0000844 | Home Loan
Sanctioned     : ₹2,842,705 | 180 months | 13.15% p.a.
EMI            : ₹36,248 | FOIR 1.00 (Financially Stressed)
Outstanding    : ₹1,068,964 | 38% utilization (Low)
Repayment Stage: Partially Repaid

[CREDIT]
Bureau Score   : 708 (Good)
Payment Trend  : Recovered From Past Delays

[DELINQUENCY]
Current DPD    : 0 days (No Delinquency) | Bucket: Current
Max DPD        : 45 days (Moderate Delinquency)

[STATUS]
Loan Status    : Active | Default: No | Write-Off: No
Risk Tier      : MEDIUM | Account Health: Healthy | Risk Driver: High EMI Burden

[SUMMARY]

EXPECTED:
LN0000844 | Home Loan | Active | MEDIUM RISK

BORROWER:

---
## Phase 1: Continued Pre-Training (CPT)

**Goal:** Make the model memorize the report's text, numbers, and concepts.

**Why this matters:** SFT alone teaches the model to answer Q&A pairs but never shows it the source text. With only 1,430 (Q,A) pairs, each fact is seen in one specific phrasing — the model memorizes that phrasing without internalizing the underlying knowledge. CPT first reads the whole report many times so the facts are encoded in weights; SFT then teaches retrieval-style behavior.

**What the math says we need:** Raw text is ~35K tokens → ~45 windows of 768 tokens. To inject knowledge: ≥300 optimizer steps. With eff_batch=8 → 50 epochs gets us ~280 steps — sufficient for memorization.

In [11]:
# ── Cell 8: Prepare CPT dataset (token-based sliding window) ─────────────────
from datasets import Dataset

with open(RAW_TEXT) as f:
    raw_text = f.read()

# Token-based chunking. The v3 char-based version cut mid-token and threw away
# structure — this version produces clean, full-token windows.
all_token_ids = tokenizer.encode(raw_text, add_special_tokens=False)
print(f"Raw text token count: {len(all_token_ids):,}")

WINDOW = 768   # tokens per training example
STRIDE = 576   # 25% overlap (192 tokens) — preserves cross-paragraph context

chunks = []
for start in range(0, len(all_token_ids), STRIDE):
    window_ids = all_token_ids[start:start + WINDOW]
    if len(window_ids) < 128:        # skip tail too small to be useful
        break
    chunks.append({"text": tokenizer.decode(window_ids, skip_special_tokens=False)})
    if start + WINDOW >= len(all_token_ids):
        break

cpt_dataset = Dataset.from_list(chunks)
print(f"CPT chunks: {len(cpt_dataset)} (window={WINDOW}, stride={STRIDE})")
print(f"\n--- First chunk preview ---\n{cpt_dataset[0]['text'][:400]}...")

Raw text token count: 13,935
CPT chunks: 24 (window=768, stride=576)

--- First chunk preview ---
LOAN PORTFOLIO & CREDIT RISK — DOMAIN KNOWLEDGE CORPUS
CPT Training Text | Retail Lending, Underwriting & Collections

This corpus covers all fields used in a retail loan dataset, including their
definitions, industry context, calculation...


In [13]:
# ── Cell 9: Run CPT ──────────────────────────────────────────────────────────
#
# Hyperparameter rationale (changes from v3 in [brackets]):
#
#   learning_rate = 2e-4         [v3: 5e-4]
#     5e-4 was too aggressive for stable memorization on tiny corpus.
#     2e-4 is the standard CPT-on-LoRA value.
#
#   num_train_epochs = 50        [v3: 10]
#     With ~45 chunks at eff_batch=8: 50 epochs ≈ 280 steps. v3's 10 epochs
#     gave only ~56 steps — far below the threshold for knowledge to stick.
#
#   per_device=2, grad_accum=4 → eff_batch=8     [v3: eff_batch=16]
#     Smaller eff_batch = more optimizer steps per epoch, which we need on
#     this small corpus.
#
#   warmup_ratio = 0.05          [v3: 0.10]
#     Total run is short; a long warmup wastes steps.
#
#   weight_decay = 0.0           [v3: 0.01]
#     We WANT memorization here. WD pulls weights toward zero — the opposite
#     of what knowledge injection needs. Save WD for SFT.
#
#   packing = True               [unchanged]
#     Correct for CPT (no response-only masking, full causal LM loss).
#
from trl import SFTTrainer, SFTConfig

cpt_trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = cpt_dataset,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs            = 10,
        learning_rate               = 2e-4,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.05,
        weight_decay                = 0.0,
        logging_steps               = 1,
        save_strategy               = "epoch",
        save_total_limit            = 2,
        optim                       = "adamw_8bit",
        packing                     = True,
        max_seq_length              = 1024,
        seed                        = 3407,
        report_to                   = "none",
        output_dir                  = CPT_DIR,
    ),
)

print("Starting CPT...")
cpt_stats = cpt_trainer.train()
print(f"\nCPT done. Final train loss: {cpt_stats.training_loss:.4f}")
print("Target: <0.5  (anything <0.7 means the model has absorbed the text)")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Sample packing skipped (custom data collator detected).


Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/24 [00:00<?, ? examples/s]

Starting CPT...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 24 | Num Epochs = 10 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.742934
2,0.754126
3,0.891361
4,0.533925
5,0.494867
6,0.434439
7,0.246984
8,0.248099
9,0.220111
10,0.121856


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-18/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-21/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-24/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-27/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /kaggle/working//qwen3_v4_cpt_ckpts/checkpoint-30/tokenizer_config.json.



CPT done. Final train loss: 0.1786
Target: <0.5  (anything <0.7 means the model has absorbed the text)


In [14]:
# ── Cell 10: Probe after CPT ─────────────────────────────────────────────────
# After CPT the model has READ the report but hasn't been taught to ANSWER
# questions about it. Outputs may be raw report-style continuations rather
# than direct answers — that's expected and fine. What you should see:
#   - Specific numbers/country names from the report appearing in outputs
#   - Vocabulary shifting toward report terminology
#   - NOT yet clean Q&A formatting
# ── Cell 7: Inference helper — baseline test on loan decision examples ────────
PROBE_EXAMPLES = [
    {
        "prompt": "[TASK] Recommend a loan approval decision: Approve | Approve with Conditions | Reject\n\n[APPLICANT]\nAge / Occupation : 65 yrs | Self Employed\nMonthly Income   : ₹93,318 | Disposable ₹0\n\n[LOAN REQUEST]\nProduct          : MSME Loan\nAmount Requested : ₹2,071,034\nTenure           : 24 months | Est. EMI ₹101,620\nFOIR             : 1.00 (Financially Stressed)\n\n[CREDIT PROFILE]\nBureau Score     : 793 (Very Good)\nPayment History  : Severely Delinquent\nCurrent DPD      : 75 days (Serious Delinquency)\nMax DPD          : 75 days (Serious Delinquency)\nCredit Util.     : 31% (Low)\n\n[FLAGS]\nDefault          : No | Write-Off: No\n\n[RECOMMENDATION]",
        "completion": "Reject"
    },
    {
        "prompt": "[TASK] Classify the credit risk of the borrower below.\nOutput one of: LOW RISK | MEDIUM RISK | HIGH RISK | CRITICAL RISK\n\n[BORROWER]\nAge / Gender   : 21 yrs | Female\nOccupation     : Salaried\nMonthly Income : ₹37,094 | Disposable ₹0\n\n[CREDITWORTHINESS]\nBureau Score   : 822 (Excellent)\nFOIR           : 1.00 (Financially Stressed)\nCredit Util.   : 31% (Low)\n\n[REPAYMENT HISTORY]\nCurrent DPD    : 0 days (No Delinquency)\nMax DPD        : 60 days (Moderate Delinquency)\nBucket         : Current\nPayment Trend  : Recovered From Past Delays\n\n[FLAGS]\nLoan Product   : Home Loan\nDefault        : No\nWrite-Off      : No\n\n[CLASSIFICATION]",
        "completion": "MEDIUM RISK\nKey signals: strong bureau score (822 (Excellent)); past delinquency (max 60 days, now recovered); stressed FOIR 1.00. Enhanced monitoring — escalate if DPD or FOIR worsens."
    },
    {
        "prompt": "[TASK] Generate a concise loan summary for a credit analyst.\n\n[BORROWER]\nCustomer       : CUST000056 | Age 44 | Male | Telangana\nOccupation     : Business\nMonthly Income : ₹30,011 | Disposable ₹0\n\n[LOAN]\nLoan ID        : LN0000844 | Home Loan\nSanctioned     : ₹2,842,705 | 180 months | 13.15% p.a.\nEMI            : ₹36,248 | FOIR 1.00 (Financially Stressed)\nOutstanding    : ₹1,068,964 | 38% utilization (Low)\nRepayment Stage: Partially Repaid\n\n[CREDIT]\nBureau Score   : 708 (Good)\nPayment Trend  : Recovered From Past Delays\n\n[DELINQUENCY]\nCurrent DPD    : 0 days (No Delinquency) | Bucket: Current\nMax DPD        : 45 days (Moderate Delinquency)\n\n[STATUS]\nLoan Status    : Active | Default: No | Write-Off: No\nRisk Tier      : MEDIUM | Account Health: Healthy | Risk Driver: High EMI Burden\n\n[SUMMARY]",
        "completion": "LN0000844 | Home Loan | Active | MEDIUM RISK\n\nBORROWER: CUST000056, 44yr male, Telangana. cyclical business income — ₹30,011/mo, disposable ₹0.\n\nLOAN: property-backed Home Loan — ₹2,842,705 over 180 months at 13.15% p.a. (EMI ₹36,248). FOIR 1.00 (Financially Stressed). Outstanding ₹1,068,964 — partially repaid, low utilization (38%). Bureau: 708 — Good.\n\nREPAYMENT: Currently regular (DPD 0); historical Max DPD 45 days (Moderate Delinquency) — Recovered From Past Delays.\n\nRISK: MEDIUM RISK — Flags: FOIR 1.00. Utilization 38% (Low). Enhanced monitoring; escalate if DPD worsens."
    },
]

def ask(prompt: str, max_new_tokens: int = 300) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
            repetition_penalty=1.1,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def probe(label: str):
    FastLanguageModel.for_inference(model)
    print(f"\n{'='*78}\n=== PROBE: {label}\n{'='*78}")
    for i, ex in enumerate(PROBE_EXAMPLES):
        print(f"\n--- Example {i+1} ---")
        print(f"PROMPT:\n{ex['prompt']}")
        print(f"\nEXPECTED:\n{ex['completion']}")
        print(f"\nMODEL OUTPUT:\n{ask(ex['prompt'], 300)}")
        print("-" * 78)
    print("=" * 78)

probe("BASELINE (no training yet)")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== PROBE: BASELINE (no training yet)

--- Example 1 ---
PROMPT:
[TASK] Recommend a loan approval decision: Approve | Approve with Conditions | Reject

[APPLICANT]
Age / Occupation : 65 yrs | Self Employed
Monthly Income   : ₹93,318 | Disposable ₹0

[LOAN REQUEST]
Product          : MSME Loan
Amount Requested : ₹2,071,034
Tenure           : 24 months | Est. EMI ₹101,620
FOIR             : 1.00 (Financially Stressed)

[CREDIT PROFILE]
Bureau Score     : 793 (Very Good)
Payment History  : Severely Delinquent
Current DPD      : 75 days (Serious Delinquency)
Max DPD          : 75 days (Serious Delinquency)
Credit Util.     : 31% (Low)

[FLAGS]
Default          : No | Write-Off: No

[RECOMMENDATION]

EXPECTED:
Reject


Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL OUTPUT:
Based on the provided details, the loan request should be rejected. Here's the rationale:

### Applicant Profile:
- Age 65 (nearly retired): Higher credit risk due to income termination.
- Self-employed: Lack of formal salary record makes income verification difficult.
- Monthly Income ₹93,318 (disposable ₹0): Zero income suggests the applicant is not
  currently generating income from the business or has stopped generating it.

### Credit Profile:
- Bureau Score 793 (Very Good): Moderate score, but in a high-risk segment like this one,
  scores need to be higher.
- Payment History: Severely delinquent (MAX DPD 75+): This is the red flag. The
  borrower has already failed to pay on previous obligations.
- Current DPD 75+: Still in default.
- Credit Utilization at 31%: While below 50%, it doesn't significantly help given
  delinquency history.
- No active trades: All trades are in NPA/CIBIL退市状态.

### Combined Analysis:
- At age 65, "unemployed" self-employed borrowers in 

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



MODEL OUTPUT:
LOW RISK
------------------------------------------------------------------------------

--- Example 3 ---
PROMPT:
[TASK] Generate a concise loan summary for a credit analyst.

[BORROWER]
Customer       : CUST000056 | Age 44 | Male | Telangana
Occupation     : Business
Monthly Income : ₹30,011 | Disposable ₹0

[LOAN]
Loan ID        : LN0000844 | Home Loan
Sanctioned     : ₹2,842,705 | 180 months | 13.15% p.a.
EMI            : ₹36,248 | FOIR 1.00 (Financially Stressed)
Outstanding    : ₹1,068,964 | 38% utilization (Low)
Repayment Stage: Partially Repaid

[CREDIT]
Bureau Score   : 708 (Good)
Payment Trend  : Recovered From Past Delays

[DELINQUENCY]
Current DPD    : 0 days (No Delinquency) | Bucket: Current
Max DPD        : 45 days (Moderate Delinquency)

[STATUS]
Loan Status    : Active | Default: No | Write-Off: No
Risk Tier      : MEDIUM | Account Health: Healthy | Risk Driver: High EMI Burden

[SUMMARY]

EXPECTED:
LN0000844 | Home Loan | Active | MEDIUM RISK

BORROWER:

---
## Phase 2: Supervised Fine-Tuning (SFT)

**Goal:** Teach the (now-knowledgeable) model to answer questions in the right format.

Same LoRA adapter as CPT — we continue training it. SFT now needs to do much less work because the knowledge is already in the weights.

In [ ]:
# ── Cell 11: Prepare SFT dataset (prompt/completion pairs) ───────────────────
import json, random
from sklearn.model_selection import train_test_split
from datasets import Dataset
random.seed(42)

with open(DATA_PATH) as f:
    raw = json.load(f)

indices = list(range(len(raw)))
train_idx, eval_idx = train_test_split(indices, test_size=0.10, random_state=42)

def fmt_prompt_completion(examples):
    texts = []
    for prompt, completion in zip(examples["prompt"], examples["completion"]):
        messages = [
            {"role": "user",      "content": prompt},
            {"role": "assistant", "content": completion},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            add_system_prompt=False,   # ← suppresses the auto Qwen system turn
        )
        texts.append(text)
    return {"text": texts}

train_records = [raw[i] for i in train_idx]
eval_records  = [raw[i] for i in eval_idx]

train_ds = Dataset.from_list(train_records).map(fmt_prompt_completion, batched=True)
eval_ds  = Dataset.from_list(eval_records).map(fmt_prompt_completion, batched=True)
print(f"SFT train: {len(train_ds)} | eval: {len(eval_ds)}")
print("\nSample formatted text:")
print(repr(train_ds[0]["text"][:300]))

In [29]:
# ── Debug Cell: Find exact boundary tokens ────────────────────────────────────
import re

sample_text = train_ds[0]["text"]

print("=== Full repr (first 500 chars) ===")
print(repr(sample_text[:500]))

print("\n=== Rendered (first 500 chars) ===")
print(sample_text[:500])

print("\n=== im_start boundaries found ===")
boundaries = re.findall(r'<\|im_start\|>\w+.{0,5}', sample_text)
for b in boundaries:
    print(repr(b))

=== Full repr (first 500 chars) ===
'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\n[TASK] Recommend a loan approval decision: Approve | Approve with Conditions | Reject\n\n[APPLICANT]\nAge / Occupation : 40 yrs | Self Employed\nMonthly Income   : ₹35,543 | Disposable ₹16,150\n\n[LOAN REQUEST]\nProduct          : Vehicle Loan\nAmount Requested : ₹607,244\nTenure           : 36 months | Est. EMI ₹19,393\nFOIR             : 0.55 (Stretched)\n\n[CREDIT PROFILE]\nBureau Score     :'

=== Rendered (first 500 chars) ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
[TASK] Recommend a loan approval decision: Approve | Approve with Conditions | Reject

[APPLICANT]
Age / Occupation : 40 yrs | Self Employed
Monthly Income   : ₹35,543 | Disposable ₹16,150

[LOAN REQUEST]
Product          : Vehicle Loan
Amount Requested : ₹607,244
Tenure           : 36 months 

In [30]:
# ── Cell 12: Custom early-stopping callback (gap-aware) ──────────────────────
from transformers import TrainerCallback

class GapMonitorEarlyStopping(TrainerCallback):
    """Stops if eval doesn't improve for `patience` evals OR if eval-train gap > threshold."""
    def __init__(self, patience=8, gap_threshold=0.6, min_evals=8):
        self.patience, self.gap_threshold, self.min_evals = patience, gap_threshold, min_evals
        self.best_eval = float("inf")
        self.no_improve = 0
        self.eval_count = 0
        self.last_train_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs and "eval_loss" not in logs:
            self.last_train_loss = logs["loss"]

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if not metrics or "eval_loss" not in metrics:
            return
        eval_loss = metrics["eval_loss"]
        self.eval_count += 1

        if eval_loss < self.best_eval - 1e-4:
            self.best_eval, self.no_improve = eval_loss, 0
        else:
            self.no_improve += 1

        gap = (eval_loss - self.last_train_loss) if self.last_train_loss is not None else None
        gap_str = f"{gap:.3f}" if gap is not None else "N/A"
        print(f"[ES] eval#{self.eval_count} eval={eval_loss:.4f} "
              f"train={self.last_train_loss or 'N/A'} gap={gap_str} "
              f"no_improve={self.no_improve}/{self.patience}")

        if self.eval_count < self.min_evals:
            return
        if self.no_improve >= self.patience:
            print(f"  ⛔ STOP: no improvement for {self.patience} evals (best={self.best_eval:.4f})")
            control.should_training_stop = True
        elif gap is not None and gap > self.gap_threshold:
            print(f"  ⛔ STOP: gap {gap:.3f} > {self.gap_threshold}")
            control.should_training_stop = True

In [ ]:
# ── Cell 13: SFT trainer ─────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

EFF_BATCH = 16
steps_per_epoch = max(1, len(train_ds) // EFF_BATCH)
print(f"Steps/epoch: ~{steps_per_epoch}, eval every 25 → ~{steps_per_epoch // 25} evals/epoch")

sft_trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        num_train_epochs            = 1,
        learning_rate               = 5e-5,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.10,
        weight_decay                = 0.05,
        logging_steps               = 1,
        eval_strategy               = "steps",
        eval_steps                  = 25,
        save_strategy               = "steps",
        save_steps                  = 25,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        optim                       = "adamw_8bit",
        packing                     = False,
        max_seq_length              = 2048,
        seed                        = 3407,
        report_to                   = "none",
        output_dir                  = SFT_DIR,
    ),
)

# ── Verify boundary tokens before applying masking ────────────────────────────
sample_text = train_ds[0]["text"]
print("=== Raw template output (first 300 chars) ===")
print(repr(sample_text[:300]))

# Qwen2.5 ChatML boundaries (no system prompt variant)
INSTRUCTION_PART = "<|im_start|>user\n"
RESPONSE_PART    = "<|im_start|>assistant\n"

# Confirm boundaries actually exist in the text
assert INSTRUCTION_PART in sample_text, \
    f"instruction_part not found! Check repr output above and update INSTRUCTION_PART."
assert RESPONSE_PART in sample_text, \
    f"response_part not found! Check repr output above and update RESPONSE_PART."
print("\n✓ Boundary tokens confirmed present in template output.")

sft_trainer = train_on_responses_only(
    sft_trainer,
    instruction_part = INSTRUCTION_PART,
    response_part    = RESPONSE_PART,
)
sft_trainer.add_callback(GapMonitorEarlyStopping(patience=8, gap_threshold=0.6, min_evals=8))

# Verify masking worked — dataset must be non-empty
assert len(sft_trainer.train_dataset) > 0, \
    "All samples masked out! Boundary tokens still wrong — check repr output above."
print(f"✓ Masking OK — {len(sft_trainer.train_dataset)} training samples retained.")

# Spot-check one sample
sample_labels = sft_trainer.train_dataset[0]["labels"]
decoded = tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in sample_labels])
print("\nMasking check (PAD = ignored, real tokens = completion only):")
print(decoded[:600])

In [ ]:
# ── Cell 14: Run SFT ─────────────────────────────────────────────────────────
sft_stats = sft_trainer.train()
print(f"\nBest checkpoint: {sft_trainer.state.best_model_checkpoint}")
print(f"Best eval_loss:  {sft_trainer.state.best_metric:.4f}")
print("\nInterpretation:")
print("  eval_loss < 0.7  → great (model knows the content well)")
print("  eval_loss 0.7-0.95 → good (model knows facts; eval set has unseen phrasings)")
print("  eval_loss > 1.1  → CPT didn't take; rerun CPT with more epochs")

In [ ]:
# ── Cell 15: Probe after SFT ─────────────────────────────────────────────────
probe("AFTER SFT (final model)")

In [ ]:
def ask1(question: str, max_new_tokens: int = 300) -> str:
    messages = [
        # {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None, top_p=None, top_k=None,
            repetition_penalty=1.1,
        )
    return tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


In [ ]:
ask("How did Sweden's economy perform in 2025 compared to what was expected in 2026?")

In [ ]:
# ── Cell 16: Plot loss curves ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

history = sft_trainer.state.log_history
tr_steps, tr_loss, ev_steps, ev_loss = [], [], [], []
for e in history:
    if 'loss' in e and 'eval_loss' not in e:
        tr_steps.append(e['step']); tr_loss.append(e['loss'])
    if 'eval_loss' in e:
        ev_steps.append(e['step']); ev_loss.append(e['eval_loss'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(tr_steps, tr_loss, label='train', color='steelblue', alpha=0.7)
axes[0].plot(ev_steps, ev_loss, label='eval',  color='tomato', linewidth=2, marker='o')
axes[0].axhline(y=0.7, color='green',  linestyle='--', alpha=0.5, label='target eval')
axes[0].set_xlabel('Steps'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].set_title('SFT loss curves')

if len(ev_steps) > 1:
    tr_interp = np.interp(ev_steps, tr_steps, tr_loss)
    gap = [e - t for e, t in zip(ev_loss, tr_interp)]
    axes[1].plot(ev_steps, gap, color='purple', linewidth=2, marker='o')
    axes[1].axhline(y=0.6, color='red',    linestyle='--', label='stop threshold')
    axes[1].axhline(y=0.4, color='orange', linestyle='--', label='warning')
    axes[1].set_xlabel('Steps'); axes[1].set_ylabel('eval - train'); axes[1].legend(); axes[1].grid(alpha=0.3)
    axes[1].set_title('Generalization gap')

plt.tight_layout()
plt.savefig(f"{SFT_DIR}/curves.png", dpi=120)
plt.show()

In [ ]:
# ── Cell 17: Quantitative evaluation on held-out questions ───────────────────
import re
from tqdm import tqdm

FastLanguageModel.for_inference(model)

def token_recall(pred, ref):
    p = set(pred.lower().split()); r = ref.lower().split()
    return sum(1 for t in r if t in p) / max(1, len(r))

def number_recall(pred, ref):
    nums = re.findall(r'\d+\.?\d*', ref)
    return 1.0 if not nums else sum(1 for n in nums if n in pred) / len(nums)

sample = eval_convs[:50]
tr, nr = [], []
for c in tqdm(sample, desc="eval"):
    pred = ask(c[1]['content'], 350)
    tr.append(token_recall(pred, c[2]['content']))
    nr.append(number_recall(pred, c[2]['content']))

print(f"\nHeld-out evaluation ({len(sample)} questions):")
print(f"  Token recall  : {sum(tr)/len(tr):.3f}  (>0.45 = good, >0.60 = excellent)")
print(f"  Number recall : {sum(nr)/len(nr):.3f}  (>0.65 = good, >0.80 = excellent)")
print("\nLowest 5 (where model fails):")
for s, c in sorted(zip(tr, sample), key=lambda x: x[0])[:5]:
    print(f"  recall={s:.2f} | {c[1]['content'][:90]}")

In [ ]:
# ── Cell 18: Save merged model ───────────────────────────────────────────────
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print(f"Saved → {MERGED_DIR}")

---
## If results are still weak after this run

Check in this order:

1. **Did CPT train_loss reach < 0.7?** If not, CPT didn't absorb the text. Solutions:
   - Increase CPT epochs to 80
   - Drop CPT eff_batch to 4 (per_device=1, grad_accum=4) → 2× more steps
   - Confirm `RAW_TEXT` actually contains the report content (not just the cover/TOC)

2. **Does the AFTER-CPT probe show report-specific vocabulary?** If output is still generic post-CPT, the model is ignoring the adapter. Solutions:
   - Bump rank to r=128 (uses ~2GB more VRAM — verify on T4)
   - Increase `lora_alpha` to 256 (effective scale doubles)

3. **Does SFT eval_loss drop below 1.0?** If not, the held-out questions test facts that even CPT didn't absorb (likely tail content). Solutions:
   - Run CPT for another 30 epochs starting from the saved CPT checkpoint
   - Reduce `test_size` from 0.10 to 0.05 (more training signal)

4. **Are inference outputs accurate but truncated?** Bump `max_new_tokens` to 500.

5. **Is the fundamental task too hard for 4B?** A 46K-token report contains thousands of facts. 4B has limited memorization capacity. Two escapes:
   - Move to Qwen3-7B or 14B (won't fit on T4 — needs ≥A100)
   - **RAG instead of fine-tuning**: embed the report with a retriever, retrieve top-k chunks at inference, let the model summarize. Much more reliable for factual QA on a fixed corpus.